# IAP · Servidor de inferencia en Colab (M1 placas) + ngrok

Levanta `inferencia/servidor/app.py` (FastAPI: YOLOv9-t ONNX + fast-plate-ocr) y lo publica con ngrok para que la web
Django (Render) lo use como `INFERENCE_URL`.

**Antes de empezar**, en el panel *Secretos* de Colab (ícono de llave, barra izquierda) crea y habilita para este notebook:

| Secreto | Obligatorio | Para qué |
|---|---|---|
| `NGROK_AUTHTOKEN` | sí | token de tu cuenta ngrok (dashboard.ngrok.com → Your Authtoken) |
| `IAP_TOKEN` | recomendado | token compartido con la web; en Render va como `INFERENCE_TOKEN`. Si no existe se genera uno |
| `NGROK_DOMINIO` | no | dominio estático gratuito de ngrok (p. ej. `algo.ngrok-free.app`): la URL no cambia entre sesiones |
| `GITHUB_TOKEN` | no | solo si el repositorio es privado |

Luego: **Entorno de ejecución → Ejecutar todo**. No hace falta GPU (todo corre en CPU con ONNX).

> Límite de ngrok gratis: cada sesión del modo caseta envía como máximo 2 frames/s × 60 s = 120 peticiones.

## 1. Configuración y secretos

In [ ]:
import os
import secrets

try:
    from google.colab import userdata
except ImportError:  # fuera de Colab: se leen variables de entorno
    userdata = None


def secreto(nombre):
    if userdata is not None:
        try:
            return userdata.get(nombre)
        except Exception:  # noqa: BLE001  (secreto inexistente o sin permiso para este notebook)
            pass
    return os.environ.get(nombre)


NGROK_AUTHTOKEN = secreto("NGROK_AUTHTOKEN")
IAP_TOKEN = secreto("IAP_TOKEN")
NGROK_DOMINIO = secreto("NGROK_DOMINIO")
GITHUB_TOKEN = secreto("GITHUB_TOKEN")

REPO = "github.com/lozogomol/IAP.git"
CARPETA = "/content/IAP"
PUERTO = 8000

assert NGROK_AUTHTOKEN, "Falta el secreto NGROK_AUTHTOKEN (panel Secretos de Colab) o no está habilitado para este notebook."
if not IAP_TOKEN:
    IAP_TOKEN = secrets.token_urlsafe(32)
    print("AVISO: No hay secreto IAP_TOKEN: se generó uno para esta sesión.")
    print("    Guárdalo como secreto IAP_TOKEN y ponlo en Render como INFERENCE_TOKEN:")
    print("   ", IAP_TOKEN)
else:
    print("[OK] Secretos cargados.")

## 2. Código del repositorio

In [ ]:
import subprocess

if os.path.isdir(f"{CARPETA}/.git"):
    subprocess.run(["git", "-C", CARPETA, "pull", "--ff-only"], check=True)
else:
    url = f"https://{GITHUB_TOKEN}@{REPO}" if GITHUB_TOKEN else f"https://{REPO}"
    subprocess.run(["git", "clone", "--depth", "1", url, CARPETA], check=True)
    # No dejar el token guardado en .git/config
    subprocess.run(["git", "-C", CARPETA, "remote", "set-url", "origin", f"https://{REPO}"], check=True)

print(subprocess.run(["git", "-C", CARPETA, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 3. Dependencias
Colab ya trae numpy y OpenCV; solo se instala lo que falta.

In [ ]:
!pip install -q fastapi uvicorn python-multipart onnxruntime "fast-plate-ocr>=1.0,<2" pyngrok

## 4. Arrancar el servidor FastAPI
Corre en segundo plano; el log queda en `/content/servidor.log`.

In [ ]:
import time
import urllib.request

try:
    servidor.terminate()  # si se vuelve a ejecutar la celda
    servidor.wait(timeout=5)
except NameError:
    pass

entorno = {**os.environ, "IAP_TOKEN": IAP_TOKEN}
log = open("/content/servidor.log", "w")
servidor = subprocess.Popen(
    ["python", "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", str(PUERTO)],
    cwd=f"{CARPETA}/inferencia/servidor", env=entorno, stdout=log, stderr=subprocess.STDOUT,
)

for _ in range(120):
    if servidor.poll() is not None:
        raise RuntimeError("El servidor se cerró. Revisa:\n" + open("/content/servidor.log").read()[-3000:])
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PUERTO}/salud", timeout=2) as r:
            print("[OK] Servidor local listo:", r.read().decode())
            break
    except Exception:  # noqa: BLE001
        time.sleep(1)
else:
    raise RuntimeError("El servidor no respondió en 2 minutos. Log:\n" + open("/content/servidor.log").read()[-3000:])

## 5. Publicar con ngrok

In [ ]:
from pyngrok import conf, ngrok

conf.get_default().auth_token = NGROK_AUTHTOKEN
ngrok.kill()  # cierra túneles de ejecuciones anteriores
opciones = {"url": "https://" + NGROK_DOMINIO} if NGROK_DOMINIO else {}
tunel = ngrok.connect(PUERTO, "http", **opciones)
URL_PUBLICA = tunel.public_url.replace("http://", "https://")

print("URL pública:", URL_PUBLICA)
print()
print("En Render → Environment pon:")
print("   INFERENCE_URL   =", URL_PUBLICA)
print("   INFERENCE_TOKEN = (el valor de IAP_TOKEN)")
if not NGROK_DOMINIO:
    print()
    print("NOTA: Sin NGROK_DOMINIO la URL cambia cada sesión: habrá que actualizar INFERENCE_URL en Render.")

## 6. Probar a través de ngrok
Comprueba `/salud` y, si quieres, sube una foto con una placa para ver la lectura.

In [ ]:
import json

import requests

CABECERAS = {"X-API-Token": IAP_TOKEN, "ngrok-skip-browser-warning": "1"}

r = requests.get(f"{URL_PUBLICA}/salud", headers=CABECERAS, timeout=15)
print("salud:", r.status_code, r.json())

r = requests.post(f"{URL_PUBLICA}/inferir", headers={"ngrok-skip-browser-warning": "1"},
                  files={"imagen": ("x.jpg", b"x", "image/jpeg")}, timeout=15)
print("sin token (debe ser 401):", r.status_code)

In [ ]:
# Opcional: subir una foto y leer la placa
from base64 import b64decode

from IPython.display import Image, display

try:
    from google.colab import files
    subidas = files.upload()
except ImportError:
    subidas = {}

for nombre, datos in subidas.items():
    r = requests.post(f"{URL_PUBLICA}/inferir", params={"anotada": "true"}, headers=CABECERAS,
                      files={"imagen": (nombre, datos, "image/jpeg")}, timeout=60)
    d = r.json()
    imagen = d.pop("imagen", None)
    print(nombre, "→", json.dumps(d, ensure_ascii=False, indent=2))
    if imagen:
        display(Image(data=b64decode(imagen.split(",", 1)[1]), width=640))

## 7. Mantener vivo y ver el log
Deja esta celda corriendo mientras usas el modo caseta. Detenla con ⏹ cuando termines.

In [ ]:
import time

from IPython.display import clear_output

try:
    while True:
        clear_output(wait=True)
        estado = "vivo" if servidor.poll() is None else f"CAÍDO (código {servidor.returncode})"
        print(f"Servidor: {estado} · URL: {URL_PUBLICA}")
        print("".join(open("/content/servidor.log").readlines()[-25:]))
        if servidor.poll() is not None:
            break
        time.sleep(10)
except KeyboardInterrupt:
    print("Monitor detenido (el servidor sigue corriendo).")

## 8. Detener

In [ ]:
ngrok.kill()
servidor.terminate()
print("Servidor y túnel detenidos.")